# Model Profiling

This notebook runs profiling for simple simulation using the [sncosmo](https://sncosmo.readthedocs.io/en/stable/)'s SALT2 model and the Rubin 10 year survey. The intention is to provide an easily reusable tool for finding bottlenecks and areas for optimization. Users looking to optimize other models, surveys, or effects, should copy this notebook and make the necessary modifications.

## Input Data

For the profiling, we use a realistic 10 year survey cadence and set of passbands for LSST. For the survey we pre-download the v5.3.0 simulated survey from [https://s3df.slac.stanford.edu/data/rubin/sim-data/](https://s3df.slac.stanford.edu/data/rubin/sim-data/). For filters we use the ones defined by the LSST preset. Both of these may need to be downloaded initially.

In [ ]:
from lightcurvelynx.astro_utils.passbands import PassbandGroup
from lightcurvelynx.obstable.opsim import OpSim
from lightcurvelynx.survey_info import SurveyInfo

# Load the OpSim data.
ops_data = OpSim.from_url(
    "https://s3df.slac.stanford.edu/data/rubin/sim-data/sims_featureScheduler_runs5.3/baseline/baseline_v5.3.0_10yrs.db",
)
t_min, t_max = ops_data.time_bounds()
print(f"Loaded OpSim with {len(ops_data)} rows and times [{t_min}, {t_max}]")

# Load the passbands.
passband_group = PassbandGroup.from_preset(preset="LSST")
print(f"Loaded passband group with {len(passband_group)} passbands.")

survey_info = SurveyInfo(obstable=ops_data, passbands=passband_group, survey_name="LSST")

## Model

For the object model, we use a SALT2 model (sncosmo's "salt2-h17") with a position that is randomly drawn from the coverage of the survey and a `t0` that is drawn uniformly from the survey's time coverage. Redshift is set randomly from 0.01 to 0.1. The remaining parameters are set as constants with: `x0`=1.0e-6, `x1`=0.5, and `c`=0.2.

Since the sncosmo models are only defined for a limited time window around t0, we extrapolate with a linear decay to zero over 50 days.

In [ ]:
from lightcurvelynx.math_nodes.np_random import NumpyRandomFunc
from lightcurvelynx.math_nodes.ra_dec_sampler import ApproximateMOCSampler
from lightcurvelynx.models.sncosmo_models import SncosmoWrapperModel
from lightcurvelynx.utils.extrapolate import LinearDecay

ra_dec_sampler = ApproximateMOCSampler.from_obstable(ops_data, depth=6)

source = SncosmoWrapperModel(
    "salt2-h17",
    t0=NumpyRandomFunc("uniform", low=t_min, high=t_max),
    x0=1.0e-6,
    x1=0.5,
    c=0.2,
    ra=ra_dec_sampler.ra,
    dec=ra_dec_sampler.dec,
    redshift=NumpyRandomFunc("uniform", low=0.01, high=0.1),
    node_label="source",
    time_extrapolation=LinearDecay(decay_width=50.0),
)

## Profiling

We perform a basic profiling using 10,000 samples of the SALT2 model over 10 years. 


In [ ]:
import cProfile

from lightcurvelynx.simulate import simulate_lightcurves

prof = cProfile.Profile()
prof.enable()
simulate_lightcurves(model=source, num_samples=10_000, survey_info=survey_info, progress_bar=False)
prof.disable()

prof.print_stats(sort="cumulative")